In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

np.set_printoptions(precision=3, suppress=True)

In [2]:
# STEP 1: Load dataset

data = load_iris()
X = data.data                      # (150, 4) feature matrix
y = data.target                    # class labels (0,1,2)
feature_names = data.feature_names
target_names = data.target_names

print("=" * 70)
print("STEP 1: Dataset")
print("=" * 70)
print("Shape of feature matrix X:", X.shape)
print("Features:", feature_names)
print("Classes:", list(target_names))
print("\nFirst 5 rows of raw data:")
print(X[:5])

STEP 1: Dataset
Shape of feature matrix X: (150, 4)
Features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Classes: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]

First 5 rows of raw data:
[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]]


In [3]:
# -----------------------------------------------------------------
# STEP 2: Standardize the data (mean = 0, std = 1)
#          PCA is variance-based, so features must be on the same scale
# -----------------------------------------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\n" + "=" * 70)
print("STEP 2: Standardized data (mean~0, std~1)")
print("=" * 70)
print("Mean after scaling:", np.round(X_scaled.mean(axis=0), 3))
print("Std  after scaling:", np.round(X_scaled.std(axis=0), 3))



STEP 2: Standardized data (mean~0, std~1)
Mean after scaling: [-0. -0. -0. -0.]
Std  after scaling: [1. 1. 1. 1.]


In [4]:
# -----------------------------------------------------------------
# STEP 3: Covariance matrix and eigen-decomposition (manual, for theory)
# -----------------------------------------------------------------
cov_matrix = np.cov(X_scaled.T)
eig_vals, eig_vecs = np.linalg.eig(cov_matrix)

# sort eigenvalues (and corresponding eigenvectors) in descending order
order = np.argsort(eig_vals)[::-1]
eig_vals = eig_vals[order]
eig_vecs = eig_vecs[:, order]

print("\n" + "=" * 70)
print("STEP 3: Covariance matrix & Eigen-decomposition")
print("=" * 70)
print("Covariance matrix:\n", cov_matrix)
print("\nEigenvalues (sorted descending):", eig_vals)
print("\nEigenvectors (columns, sorted to match eigenvalues):\n", eig_vecs)


STEP 3: Covariance matrix & Eigen-decomposition
Covariance matrix:
 [[ 1.007 -0.118  0.878  0.823]
 [-0.118  1.007 -0.431 -0.369]
 [ 0.878 -0.431  1.007  0.969]
 [ 0.823 -0.369  0.969  1.007]]

Eigenvalues (sorted descending): [2.938 0.92  0.148 0.021]

Eigenvectors (columns, sorted to match eigenvalues):
 [[ 0.521 -0.377 -0.72   0.261]
 [-0.269 -0.923  0.244 -0.124]
 [ 0.58  -0.024  0.142 -0.801]
 [ 0.565 -0.067  0.634  0.524]]


In [14]:
X_scaled.shape

(150, 4)

In [5]:
# -----------------------------------------------------------------
# STEP 4: Fit PCA using scikit-learn (all components first, to inspect variance)
# -----------------------------------------------------------------
pca_full = PCA(n_components=X_scaled.shape[1])
X_pca_full = pca_full.fit_transform(X_scaled)

explained_var = pca_full.explained_variance_ratio_
cum_explained_var = np.cumsum(explained_var)

print("\n" + "=" * 70)
print("STEP 4: Explained Variance Ratio (sklearn PCA, all components)")
print("=" * 70)
for i, (v, c) in enumerate(zip(explained_var, cum_explained_var)):
    print(f"PC{i+1}: explained variance = {v*100:5.2f}%   cumulative = {c*100:5.2f}%")



STEP 4: Explained Variance Ratio (sklearn PCA, all components)
PC1: explained variance = 72.96%   cumulative = 72.96%
PC2: explained variance = 22.85%   cumulative = 95.81%
PC3: explained variance =  3.67%   cumulative = 99.48%
PC4: explained variance =  0.52%   cumulative = 100.00%


In [7]:
# -----------------------------------------------------------------
# STEP 5: Scree plot (explained variance per component)
# -----------------------------------------------------------------
plt.figure(figsize=(7, 4.5))
components = np.arange(1, len(explained_var) + 1)
plt.bar(components, explained_var * 100, color="#4C72B0", alpha=0.85, label="Individual")
plt.plot(components, cum_explained_var * 100, color="#C44E52", marker="o", label="Cumulative")
plt.axhline(y=95, color="gray", linestyle="--", linewidth=1, label="95% threshold")
plt.xticks(components, [f"PC{i}" for i in components])
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance (%)")
plt.title("Scree Plot - Explained Variance by Component")
plt.legend()
plt.tight_layout()
plt.savefig("scree_plot.png", dpi=150)
plt.close()
print("\nScree plot saved -> scree_plot.png")


Scree plot saved -> scree_plot.png


In [9]:
# -----------------------------------------------------------------
# STEP 6: Reduce to 2 components and visualize
# -----------------------------------------------------------------
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

print("\n" + "=" * 70)
print("STEP 6: PCA reduced to 2 components")
print("=" * 70)
print("Shape after PCA:", X_pca_2d.shape)
print(f"Variance explained by PC1+PC2: {sum(pca_2d.explained_variance_ratio_)*100:.2f}%")

colors = ["#4C72B0", "#DD8452", "#55A868"]
plt.figure(figsize=(7, 5.5))
for i, name in enumerate(target_names):
    mask = y == i
    plt.scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1],
                label=name, color=colors[i], alpha=0.8, edgecolor="white", s=60)
plt.xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.title("PCA-Transformed Iris Data (2 Components)")
plt.legend(title="Species")
plt.tight_layout()
plt.savefig("pca_2d_scatter.png", dpi=150)
plt.close()
print("2D PCA scatter plot saved -> pca_2d_scatter.png")


STEP 6: PCA reduced to 2 components
Shape after PCA: (150, 2)
Variance explained by PC1+PC2: 95.81%
2D PCA scatter plot saved -> pca_2d_scatter.png


In [11]:
# -----------------------------------------------------------------
# STEP 7: 3-component PCA and 3D visualization
# -----------------------------------------------------------------
pca_3d = PCA(n_components=3)
X_pca_3d = pca_3d.fit_transform(X_scaled)

fig = plt.figure(figsize=(7.5, 6))
ax = fig.add_subplot(111, projection="3d")
for i, name in enumerate(target_names):
    mask = y == i
    ax.scatter(X_pca_3d[mask, 0], X_pca_3d[mask, 1], X_pca_3d[mask, 2],
               label=name, color=colors[i], alpha=0.8, s=45)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
ax.set_title("PCA-Transformed Iris Data (3 Components)")
ax.legend(title="Species")
plt.tight_layout()
plt.savefig("pca_3d_scatter.png", dpi=150)
plt.close()
print(f"3D PCA scatter saved -> pca_3d_scatter.png "
      f"(cumulative variance PC1-3: {sum(pca_3d.explained_variance_ratio_)*100:.2f}%)")



3D PCA scatter saved -> pca_3d_scatter.png (cumulative variance PC1-3: 99.48%)


In [12]:
# -----------------------------------------------------------------
# STEP 8: Choosing number of components for 95% variance retained
# -----------------------------------------------------------------
pca_95 = PCA(n_components=0.95)
X_pca_95 = pca_95.fit_transform(X_scaled)

print("\n" + "=" * 70)
print("STEP 9: Components needed to retain >= 95% variance")
print("=" * 70)
print(f"Number of components selected: {pca_95.n_components_}")
print(f"Total variance retained: {sum(pca_95.explained_variance_ratio_)*100:.2f}%")

print("\nDone.")



STEP 9: Components needed to retain >= 95% variance
Number of components selected: 2
Total variance retained: 95.81%

Done.


In [23]:
# -----------------------------------------------------------------
# STEP 8: Choosing number of components for 95% variance retained
# -----------------------------------------------------------------
pca_95 = PCA(n_components=0.995)
X_pca_95 = pca_95.fit_transform(X_scaled)

print("\n" + "=" * 70)
print("STEP 9: Components needed to retain >= 95% variance")
print("=" * 70)
print(f"Number of components selected: {pca_95.n_components_}")
print(f"Total variance retained: {sum(pca_95.explained_variance_ratio_)*100:.2f}%")

print("\nDone.")


STEP 9: Components needed to retain >= 95% variance
Number of components selected: 4
Total variance retained: 100.00%

Done.
